[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/22_robotics_action_policies.ipynb)

# 22. Robotics action policies — paper-faithful tiny structures

이전 버전은 ACT를 position embedding을 더한 MLP 수준으로 줄였고, Diffusion Policy도 실제 conditional denoising network 없이 clean-action 복원식 하나만 있었다.

이번 버전은 다음 핵심 구조를 보존한다.

- ACT: **action chunk + CVAE latent + Transformer query decoding**
- Diffusion Policy: **observation-conditioned action horizon denoising + timestep conditioning + iterative sampling**
- π0-style flow policy: **noisy action horizon의 velocity field를 여러 ODE step으로 적분**
- FAST: scalar binning 대신 **action trajectory DCT → quantization → token sequence**의 핵심 전처리


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Behavior cloning baseline

가장 단순한 BC는 현재 observation 하나에서 action 하나를 직접 회귀한다. 이 baseline과 비교해야 action chunking이나 generative action policy가 무엇을 추가하는지 보인다.


In [ ]:
observation = torch.randn(4, 8, device=device)
target_action = torch.randn(4, 3, device=device)

bc_policy = nn.Linear(8, 3).to(device)
predicted_action = bc_policy(observation)
bc_loss = F.mse_loss(predicted_action, target_action)

print("BC loss:", bc_loss.item())


## 2. ACT: action chunk + CVAE + Transformer decoder

ACT는 단순히 한 번에 여러 action을 Linear layer로 출력하는 것이 아니다. 학습 시 demonstration action chunk를 encoder가 읽어 latent `z`의 distribution을 만들고, decoder는 observation/state와 latent를 condition으로 받아 **여러 learned action query**에서 미래 action chunk를 동시에 생성한다.

아래 tiny model은 image backbone 대신 이미 만들어진 observation token을 사용하지만, **CVAE style encoder와 Transformer query decoder**는 남긴다.


In [ ]:
class TinyACT(nn.Module):
    def __init__(
        self,
        observation_dim=8,
        action_dim=3,
        chunk_size=4,
        hidden_dim=24,
        latent_dim=8,
    ):
        super().__init__()

        self.chunk_size = chunk_size
        self.action_dim = action_dim

        self.observation_projection = nn.Linear(
            observation_dim,
            hidden_dim,
        )
        self.action_projection = nn.Linear(
            action_dim,
            hidden_dim,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=3,
            dim_feedforward=4 * hidden_dim,
            batch_first=True,
        )
        self.style_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1,
        )
        self.mu_head = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)

        self.latent_projection = nn.Linear(
            latent_dim,
            hidden_dim,
        )
        self.action_queries = nn.Parameter(
            torch.randn(1, chunk_size, hidden_dim) * 0.02
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=3,
            dim_feedforward=4 * hidden_dim,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=1,
        )
        self.action_head = nn.Linear(hidden_dim, action_dim)

    def encode_style(self, observation, action_chunk):
        observation_token = self.observation_projection(
            observation
        ).unsqueeze(1)
        action_tokens = self.action_projection(action_chunk)

        encoder_input = torch.cat(
            [observation_token, action_tokens],
            dim=1,
        )
        encoded = self.style_encoder(encoder_input)
        style_token = encoded[:, 0]

        mu = self.mu_head(style_token)
        logvar = self.logvar_head(style_token)
        return mu, logvar

    def forward(self, observation, action_chunk=None):
        batch_size = observation.size(0)

        if action_chunk is not None:
            mu, logvar = self.encode_style(
                observation,
                action_chunk,
            )
            std = torch.exp(0.5 * logvar)
            latent = mu + std * torch.randn_like(std)
        else:
            mu = torch.zeros(batch_size, 8, device=observation.device)
            logvar = torch.zeros_like(mu)
            latent = torch.zeros_like(mu)

        observation_token = self.observation_projection(
            observation
        ).unsqueeze(1)
        latent_token = self.latent_projection(latent).unsqueeze(1)
        memory = torch.cat(
            [observation_token, latent_token],
            dim=1,
        )

        queries = self.action_queries.expand(
            batch_size,
            -1,
            -1,
        )
        decoded = self.decoder(
            tgt=queries,
            memory=memory,
        )

        predicted_chunk = self.action_head(decoded)
        return predicted_chunk, mu, logvar


act = TinyACT().to(device)
target_chunk = torch.randn(4, 4, 3, device=device)

predicted_chunk, mu, logvar = act(
    observation,
    target_chunk,
)

reconstruction_loss = F.l1_loss(
    predicted_chunk,
    target_chunk,
)
kl_loss = -0.5 * torch.mean(
    1 + logvar - mu.square() - logvar.exp()
)

print("predicted chunk:", predicted_chunk.shape)
print("reconstruction loss:", reconstruction_loss.item())
print("KL loss:", kl_loss.item())


## 3. Diffusion Policy: condition an action horizon, not one action

Diffusion Policy는 observation을 condition으로 사용해 **action sequence 전체의 noisy trajectory**를 반복적으로 denoise한다. 중요한 점은 action horizon과 receding-horizon execution이다.

아래 tiny denoiser는 action sequence를 1D temporal convolution으로 처리하고 timestep embedding과 observation condition을 더한다.


In [ ]:
def timestep_embedding(t, hidden_dim):
    half = hidden_dim // 2
    frequencies = torch.exp(
        -math.log(10000)
        * torch.arange(half, device=t.device)
        / half
    )
    angles = t[:, None] * frequencies[None]
    return torch.cat([angles.sin(), angles.cos()], dim=-1)


class TinyDiffusionPolicy(nn.Module):
    def __init__(self, observation_dim=8, action_dim=3, hidden_dim=32):
        super().__init__()

        self.input_projection = nn.Conv1d(
            action_dim,
            hidden_dim,
            kernel_size=1,
        )
        self.temporal_block = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1),
            nn.SiLU(),
            nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1),
        )
        self.observation_projection = nn.Linear(
            observation_dim,
            hidden_dim,
        )
        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.output_projection = nn.Conv1d(
            hidden_dim,
            action_dim,
            kernel_size=1,
        )

    def forward(self, noisy_actions, observation, t):
        hidden = self.input_projection(
            noisy_actions.transpose(1, 2)
        )

        observation_condition = self.observation_projection(
            observation
        )[:, :, None]
        time_condition = self.time_mlp(
            timestep_embedding(t, hidden.size(1))
        )[:, :, None]

        hidden = hidden + observation_condition + time_condition
        hidden = hidden + self.temporal_block(hidden)

        predicted_noise = self.output_projection(hidden)
        return predicted_noise.transpose(1, 2)


diffusion_policy = TinyDiffusionPolicy().to(device)
action_horizon = 6
clean_actions = torch.randn(4, action_horizon, 3, device=device)
noise = torch.randn_like(clean_actions)
t = torch.rand(4, device=device)

alpha = (1 - 0.8 * t)[:, None, None]
sigma = torch.sqrt(1 - alpha.square())
noisy_actions = alpha * clean_actions + sigma * noise

predicted_noise = diffusion_policy(
    noisy_actions,
    observation,
    t,
)
diffusion_loss = F.mse_loss(predicted_noise, noise)

print("action horizon:", clean_actions.shape)
print("predicted noise:", predicted_noise.shape)
print("diffusion loss:", diffusion_loss.item())


## 4. Diffusion Policy sampling and receding horizon

실행 시에는 noise action horizon에서 시작해 여러 denoising step을 거쳐 action sequence를 만든 뒤, 전체 horizon을 한꺼번에 blindly 실행하지 않고 앞부분만 실행하고 다음 observation에서 다시 계획하는 receding-horizon 방식으로 사용한다.


In [ ]:
sample = torch.randn(
    1, action_horizon, 3,
    device=device,
)
condition = observation[:1]

with torch.no_grad():
    for step in reversed(range(5)):
        t_value = torch.full(
            (1,),
            step / 4,
            device=device,
        )
        predicted_noise = diffusion_policy(
            sample,
            condition,
            t_value,
        )
        sample = sample - 0.15 * predicted_noise

execute_now = sample[:, :2]

print("planned horizon:", sample.shape)
print("executed prefix:", execute_now.shape)


## 5. Flow-matching action policy

π0 계열의 action generation은 noisy action horizon에서 expert action horizon으로 가는 velocity field를 학습하고 ODE처럼 여러 step 적분한다. VLA의 vision/language joint attention 구조는 03번에서 별도로 구현했으므로 여기서는 action-space flow 자체에 집중한다.


In [ ]:
noise_actions = torch.randn(2, 4, 3, device=device)
expert_actions = torch.randn(2, 4, 3, device=device)
t = torch.rand(2, device=device)
t_broadcast = t[:, None, None]

action_t = (
    (1 - t_broadcast) * expert_actions
    + t_broadcast * noise_actions
)
target_velocity = noise_actions - expert_actions

print("flow input:", action_t.shape)
print("velocity target:", target_velocity.shape)

sample = noise_actions[:1].clone()

def oracle_velocity(current_action, time):
    return expert_actions[:1] - current_action

with torch.no_grad():
    for step in range(6):
        time = torch.full(
            (1,),
            step / 6,
            device=device,
        )
        sample = sample + (1 / 6) * oracle_velocity(sample, time)

print("flow-sampled action horizon:", sample.shape)


## 6. FAST action tokenization: DCT before discrete tokens

FAST는 continuous action 값을 각 시점별 scalar bin으로 바로 바꾸는 방식이 아니다. action trajectory를 dimension별 DCT coefficient로 바꾸고, coefficient를 quantize한 뒤 low-frequency components가 먼저 오도록 flatten한다. 이후 실제 FAST에서는 BPE tokenizer가 이 integer sequence를 losslessly 압축해 VLM vocabulary에 넣을 action token sequence를 만든다.


In [ ]:
def dct_matrix(length, device):
    n = torch.arange(length, device=device).float()
    k = torch.arange(length, device=device).float()[:, None]

    matrix = torch.cos(
        math.pi / length
        * (n[None, :] + 0.5)
        * k
    )
    matrix[0] *= 1 / math.sqrt(2)
    matrix *= math.sqrt(2 / length)
    return matrix


action_trajectory = torch.tensor(
    [
        [0.0, 0.0],
        [0.2, 0.1],
        [0.4, 0.1],
        [0.6, 0.2],
        [0.8, 0.2],
        [1.0, 0.3],
    ],
    device=device,
)

DCT = dct_matrix(
    action_trajectory.size(0),
    device,
)
coefficients = DCT @ action_trajectory

quantization_scale = 100.0
quantized = torch.round(
    coefficients * quantization_scale
).to(torch.int64)

# Frequency-major flattening: low frequency coefficients appear first.
integer_sequence = quantized.reshape(-1)

print("DCT coefficients:\n", coefficients)
print("quantized coefficients:\n", quantized)
print("pre-BPE integer sequence:", integer_sequence)


## References and provenance

**ACT** — Zhao et al., *Learning Fine-Grained Bimanual Manipulation with Low-Cost Hardware* 및 공식 `tonyzhaozh/act`. action chunking, CVAE latent, Transformer query decoding을 반영했다.

**Diffusion Policy** — Chi et al., *Diffusion Policy: Visuomotor Policy Learning via Action Diffusion*. observation-conditioned action-horizon denoising, temporal model, iterative sampling, receding-horizon execution을 반영했다. 실제 논문의 구현은 더 큰 temporal U-Net/Transformer와 diffusion schedule을 사용한다.

**π0 / openpi** — Physical Intelligence π0 family. action-horizon flow matching과 iterative integration을 반영했으며 vision/language/action joint attention은 03번에 구현했다.

**FAST** — Physical Intelligence, *FAST: Efficient Action Tokenization for Vision-Language-Action Models*. DCT coefficients → quantization → frequency-major sequence → BPE compression이라는 핵심 구조를 반영했다. 이 노트북은 BPE vocabulary 학습 자체는 생략하고 BPE 입력 직전 integer sequence까지 보여준다.
